# Endometrium scRNA-seq Analysis Pipeline
**Author:** Shen Khaw
**Date:** 6/11/26
**Description:** This file contains the full pipeline for QC, SCVI integration using raw counts, SCANVI cell-type prediction, and downstream transcription factor analysis.

In [ ]:
# Configuration and Imports
import scanpy as sc
import scvi
import pandas as pd
import matplotlib.pyplot as plt
import os

# Set plotting parameters for publication-quality figures
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=300, facecolor='white')

## 1. Data Loading
Loading the dataset integrated using the corrected SCVI pipeline (using `layer="counts"` to ensure proper Negative Binomial distribution).

In [ ]:
import os
from pathlib import Path
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Get the directory where the notebook is currently running
BASE_DIR = Path.cwd() 

# 2. Define the input file relative to the notebook
INPUT_FILE = BASE_DIR / "endometrium_final.h5ad"

# 3. Create the Figures folder automatically
FIG_DIR = BASE_DIR / "Figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Working directory: {BASE_DIR}")
print(f"📁 Figure output folder: {FIG_DIR}")

# DATA LOADING & PROCESSING
# 1. Load the file
adata = sc.read_h5ad(INPUT_FILE)

# 2. Safely store raw counts if we haven't already
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

# 3. FIX: Reset adata.X to the RAW counts to start clean
adata.X = adata.layers["counts"].copy()

# 4. Now safely normalize and log-transform EXACTLY once
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 5. Compute UMAP from scANVI latent space
sc.pp.neighbors(adata, use_rep="X_scANVI")
sc.tl.umap(adata)

print("✅ Data reset to raw counts, normalized once, and UMAP ready.")

## 2. Quality Control (QC)

In [ ]:
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Removed the sc.read_h5ad() so it doesn't overwrite our UMAP/normalization!

qc_cols = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']

# We use the adata that is already in memory
df_all = adata.obs[qc_cols + ['sample_id']].copy()
df_all = df_all.rename(columns={'sample_id': 'sample'}) # Modern pandas way

print(f"✅ Extracted {len(df_all):,} cells.")
print(f"Samples found: {df_all['sample'].nunique()}")

## 3. UMAP Visualization
Visualizing the SCANVI latent space to verify batch correction and cell-type clustering.

In [ ]:
import scanpy as sc

# Plot UMAP colored by cell type and dataset
sc.pl.umap(adata, color=['scanvi_prediction'], title="SCANVI Predictions", frameon=False)
sc.pl.umap(adata, color=['sample_id'], title="Batch Correction Check", frameon=False)

In [ ]:
import math
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

db_groups = {
    "GSE183837":    ["GSE183837_Ctrl-1", "GSE183837_Ctrl-2", "GSE183837_Ctrl-3"],
    "GSE214411":    ["GSE214411_N5", "GSE214411_N7", "GSE214411_N1"],
    "E-MTAB-10287": ["E-MTAB-10287_MRC_Endo8712024"],
    "E-MTAB-14039": ["E-MTAB-14039_FX1106", "E-MTAB-14039_FX9011"],
    "GSE250130":    ["GSE250130_E03", "GSE250130_E04", "GSE250130_E08",
                     "GSE250130_E09", "GSE250130_E11", "GSE250130_E14"],
}

_FIXED_COLORS = {
    "endothelial": "#F39C12",
    "epithelial": "#3498DB",
    "immune": "#2ECC71",
    "stromal": "#E74C3C"
}
_FALLBACK_COLORS = ["#9B59B6", "#1ABC9C", "#34495E", "#E67E22", "#7F8C8D"]

cats = adata.obs['scanvi_prediction'].cat.categories
color_of = {t: _FIXED_COLORS.get(str(t).strip().lower(), _FALLBACK_COLORS[i % len(_FALLBACK_COLORS)]) for i, t in enumerate(cats)}

for db, samples in db_groups.items():
    valid_samples = [s for s in samples if s in adata.obs['sample_id'].values]
    
    if not valid_samples:
        continue

    n = len(valid_samples)
    ncols = min(n, 3) 
    nrows = math.ceil(n / ncols)
    figsize = (5.5 * ncols + 2.5, 5 * nrows)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    axes = axes.flatten()

    for ax, samp in zip(axes, valid_samples):
        sub = adata[adata.obs['sample_id'] == samp]
        present_types = sub.obs['scanvi_prediction'].unique()
        
        sc.pl.umap(
            sub, 
            color='scanvi_prediction', 
            ax=ax, 
            show=False, 
            size=25,             
            title=str(samp),
            frameon=True,        
            legend_loc=None,
            palette={t: color_of[t] for t in present_types}
        )
        
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('equal')
        ax.set_title(str(samp), fontsize=12, pad=10)

    for ax in axes[n:]:
        ax.set_visible(False)

    handles = [mpatches.Patch(color=color_of[k], label=k) for k in cats]
    fig.legend(
        handles=handles, 
        loc="center right", 
        title="Cell Type", 
        bbox_to_anchor=(0.98, 0.5),
        frameon=False,
        fontsize=11, 
        title_fontsize=12
    )
    
    fig.suptitle(f"Dataset: {db} — SCANVI Predictions per Sample", fontsize=16, fontweight="bold")
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.88, right=0.85) 
    
    # --- NEW: Save the figures to the folder ---
    save_path_pdf = os.path.join(FIG_DIR, f"UMAP_{db}_per_sample.pdf")
    save_path_png = os.path.join(FIG_DIR, f"UMAP_{db}_per_sample.png")
    
    # bbox_inches='tight' ensures no labels get cut off when saving
    plt.savefig(save_path_pdf, bbox_inches='tight')
    plt.savefig(save_path_png, dpi=300, bbox_inches='tight')
    
    print(f"✅ Saved: {save_path_png}")
    
    plt.show()
    plt.close(fig)

## 4. Dotplots

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

markers = {
    "Stromal": ["DCN", "LUM", "COL1A1", "COL1A2", "PDGFRA"],
    "Epithelial": ["EPCAM", "KRT8", "KRT18", "KRT19", "CDH1"],
    "Immune": ["PTPRC"],
    "Endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5", "ERG"]
}
valid_markers = {k: [g for g in v if g in adata.var_names] for k, v in markers.items()}

# ==========================================
# PART 1: THE OVERALL DOT PLOT
# ==========================================
print("Generating Overall Dot Plot...")
sc.pl.dotplot(
    adata, 
    valid_markers, 
    groupby='scanvi_prediction', 
    standard_scale='var', 
    cmap='Reds',
    show=False
)

# FIX: Grab the figure, add a clean title, and adjust spacing
fig = plt.gcf()
fig.suptitle('Overall Canonical Marker Expression', fontsize=16, fontweight='bold', y=1.05)
fig.subplots_adjust(top=0.85) # Pushes the plot down to leave room for labels + title

fig.savefig(os.path.join(FIG_DIR, "DotPlot_Overall.pdf"), bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, "DotPlot_Overall.png"), dpi=300, bbox_inches='tight')
plt.close(fig) 
print("✅ Saved: DotPlot_Overall")


# ==========================================
# PART 2: INDIVIDUAL DOT PLOTS PER SAMPLE
# ==========================================
print("\nGenerating Individual Dot Plots...")
samples = adata.obs['sample_id'].unique()

for samp in samples:
    sub = adata[adata.obs['sample_id'] == samp]
    
    sc.pl.dotplot(
        sub, 
        valid_markers, 
        groupby='scanvi_prediction', 
        standard_scale='var', 
        cmap='Reds', 
        show=False
    )
    
    safe_samp = str(samp).replace(" ", "_").replace("/", "_")
    
    # FIX: Same layout adjustments for individual plots
    fig = plt.gcf()
    fig.suptitle(f'{samp} - Canonical Markers', fontsize=14, fontweight='bold', y=1.05)
    fig.subplots_adjust(top=0.85)
    
    fig.savefig(os.path.join(FIG_DIR, f"DotPlot_{safe_samp}.pdf"), bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, f"DotPlot_{safe_samp}.png"), dpi=300, bbox_inches='tight')
    
    plt.close(fig) 
    print(f"✅ Saved: DotPlot_{safe_samp}")

print(f"\n🎉 All Dot Plots successfully saved to the '{FIG_DIR}' folder!")

## 5. TF Specificity, Heatmap & Volcano plot

In [ ]:
# 5A. Generate sample-level pseudobulk profiles from raw counts
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse

PSEUDOBULK_DIR = BASE_DIR / "Pseudobulk"
PSEUDOBULK_DIR.mkdir(parents=True, exist_ok=True)

MIN_CELLS_PER_PSEUDOBULK = 10
CELLTYPE_COL = "scanvi_prediction"
SAMPLE_COL = "sample_id"

required_obs = [SAMPLE_COL, CELLTYPE_COL]
missing_obs = [col for col in required_obs if col not in adata.obs.columns]
if missing_obs:
    raise KeyError(f"Missing required adata.obs columns: {missing_obs}")

if "counts" not in adata.layers:
    raise KeyError("Raw counts are required in adata.layers['counts'].")

# Keep gene names unique so the pseudobulk matrix has unambiguous columns.
adata.var_names_make_unique()
counts = adata.layers["counts"]

# Validate the raw-count layer before aggregation.
if sparse.issparse(counts):
    count_values = counts.data
else:
    count_values = np.asarray(counts).ravel()

if count_values.size and (
    not np.isfinite(count_values).all()
    or (count_values < 0).any()
    or not np.allclose(count_values, np.round(count_values), atol=1e-6)
):
    raise ValueError(
        "The counts layer is not non-negative integer-valued raw counts. "
        "Do not generate pseudobulk results from normalized/log-transformed expression."
    )

# Work with canonical cell-type labels, but retain the original labels in adata.obs.
def canonical_celltype(value):
    value = str(value).strip().lower()
    mapping = {
        "stromal": "Stromal",
        "epithelial": "Epithelial",
        "immune": "Immune",
        "endothelial": "Endothelial",
    }
    return mapping.get(value, str(value).strip())

obs = adata.obs[[SAMPLE_COL, CELLTYPE_COL]].copy()
obs[SAMPLE_COL] = obs[SAMPLE_COL].astype(str)
obs["_cell_type"] = obs[CELLTYPE_COL].map(canonical_celltype)

pb_count_columns = {}
pb_metadata_rows = []

grouped = obs.groupby([SAMPLE_COL, "_cell_type"], observed=True, sort=True).indices

for (sample_id, cell_type), positions in grouped.items():
    positions = np.asarray(positions, dtype=int)
    n_cells = len(positions)

    if n_cells < MIN_CELLS_PER_PSEUDOBULK:
        print(f"Skipping {sample_id} / {cell_type}: {n_cells} cells (<{MIN_CELLS_PER_PSEUDOBULK})")
        continue

    group_counts = counts[positions]
    summed = np.asarray(group_counts.sum(axis=0)).ravel().astype(np.float64)

    total_counts = float(summed.sum())
    if total_counts <= 0:
        print(f"Skipping {sample_id} / {cell_type}: zero total counts")
        continue

    pseudobulk_id = f"{sample_id}|{cell_type}"
    pb_count_columns[pseudobulk_id] = pd.Series(summed, index=adata.var_names)
    pb_metadata_rows.append({
        "pseudobulk_sample": pseudobulk_id,
        "sample_id": sample_id,
        "cell_type": cell_type,
        "n_cells": n_cells,
        "total_counts_raw": total_counts,
    })

if not pb_count_columns:
    raise ValueError(
        "No pseudobulk groups passed the minimum-cell threshold. "
        "Check sample IDs, cell-type labels, and MIN_CELLS_PER_PSEUDOBULK."
    )

# Rows = genes; columns = sample-by-cell-type pseudobulk profiles.
pb_counts = pd.DataFrame(pb_count_columns).fillna(0)
pb_metadata = pd.DataFrame(pb_metadata_rows)

# Preserve summed raw counts for any later count-based modelling.
pb_counts_path = PSEUDOBULK_DIR / "pseudobulk_raw_counts.csv"
pb_metadata_path = PSEUDOBULK_DIR / "pseudobulk_metadata.csv"
pb_counts.to_csv(pb_counts_path, index_label="gene")
pb_metadata.to_csv(pb_metadata_path, index=False)

# For the exploratory specificity ranking, calculate CPM-like expression
# separately within each sample-by-cell-type pseudobulk profile.
pb_norm = pb_counts.div(pb_counts.sum(axis=0).replace(0, np.nan), axis=1) * 10_000
pb_norm = pb_norm.fillna(0)
pb_norm_path = PSEUDOBULK_DIR / "pseudobulk_normalized_per_10k.csv"
pb_norm.to_csv(pb_norm_path, index_label="gene")

print(f"Saved raw pseudobulk counts: {pb_counts.shape[0]:,} genes × {pb_counts.shape[1]:,} profiles")
print(f"Saved metadata: {len(pb_metadata):,} pseudobulk profiles")
print("\nPseudobulk profiles per cell type:")
print(pb_metadata["cell_type"].value_counts())
print(f"\nFiles saved in: {PSEUDOBULK_DIR}")

In [ ]:
# 5B. Sample-paired TF specificity and importance analysis

from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests

print("Running sample-paired TF specificity analysis...")

# 1. Prepare pseudobulk metadata
meta = pb_metadata.set_index("pseudobulk_sample")
meta = meta.loc[pb_norm.columns].copy()

required_columns = ["sample_id", "cell_type"]
missing_columns = [c for c in required_columns if c not in meta.columns]

if missing_columns:
    raise ValueError(f"Missing metadata columns: {missing_columns}")

# Prevent silent overwriting of pseudobulk profiles.
duplicate_profiles = (
    meta.reset_index()
    .groupby(["sample_id", "cell_type"], observed=True)
    .size()
)

duplicate_profiles = duplicate_profiles[duplicate_profiles > 1]

if not duplicate_profiles.empty:
    raise ValueError(
        "Multiple pseudobulk profiles were found for the same "
        "sample and cell type. Resolve these before continuing:\n"
        f"{duplicate_profiles}"
    )

TARGET_CELL_TYPES = ["Stromal", "Epithelial"]
results = []

# 2. Calculate TF expression, AUC and paired statistical tests
for target_ct in TARGET_CELL_TYPES:

    print(f"\nAnalysing {target_ct} cells...")

    target_meta = meta[meta["cell_type"] == target_ct]
    other_meta = meta[meta["cell_type"] != target_ct]

    paired_samples = sorted(
        set(target_meta["sample_id"]) &
        set(other_meta["sample_id"])
    )

    if len(paired_samples) < 2:
        print(f"Skipping {target_ct}: fewer than two matched samples.")
        continue

    # One pseudobulk profile per sample and target cell type.
    target_cols = {
        row["sample_id"]: idx
        for idx, row in target_meta.iterrows()
        if row["sample_id"] in paired_samples
    }

    # Other cell-type profiles from the same biological sample.
    other_cols_by_sample = {
        sample: other_meta.index[
            other_meta["sample_id"] == sample
        ].tolist()
        for sample in paired_samples
    }

    # Calculate mean expression across ALL cell types.
    # Use the same matched sample set for this target analysis.

    all_celltype_means = {}

    for ct in meta["cell_type"].dropna().unique():

        ct_cols = meta.index[
            (meta["cell_type"] == ct) &
            (meta["sample_id"].isin(paired_samples))
        ].tolist()

        if ct_cols:
            all_celltype_means[ct] = (
                pb_norm.loc[tf_genes, ct_cols].mean(axis=1)
            )

    all_celltype_mean_df = pd.DataFrame(all_celltype_means)

    all_celltype_total = (
        all_celltype_mean_df.sum(axis=1).replace(0, np.nan)
    )

    for tf in tf_genes:

        target_values = []
        other_values = []

        for sample in paired_samples:

            target_col = target_cols.get(sample)
            other_cols = other_cols_by_sample.get(sample, [])

            if target_col is None or not other_cols:
                continue

            target_values.append(
                float(pb_norm.at[tf, target_col])
            )

            # Mean of non-target cell types within the same sample.
            other_values.append(
                float(pb_norm.loc[tf, other_cols].mean())
            )

        target_values = np.asarray(target_values, dtype=float)
        other_values = np.asarray(other_values, dtype=float)

        valid = (
            np.isfinite(target_values) &
            np.isfinite(other_values)
        )

        target_values = target_values[valid]
        other_values = other_values[valid]

        if len(target_values) < 2:
            continue

        mean_target = float(np.mean(target_values))
        mean_other = float(np.mean(other_values))

        log2fc = float(
            np.log2((mean_target + 1e-3) / (mean_other + 1e-3))
        )

        # Descriptive AUC: target versus matched reference profiles.
        try:
            labels = np.r_[
                np.ones(len(target_values)),
                np.zeros(len(other_values))
            ]

            scores = np.r_[target_values, other_values]

            auc = float(roc_auc_score(labels, scores))

        except Exception:
            auc = 0.5

        # Paired one-sided Wilcoxon signed-rank test.
        try:
            if np.allclose(target_values, other_values):
                pval = 1.0
            else:
                pval = float(
                    wilcoxon(
                        target_values,
                        other_values,
                        alternative="greater",
                        zero_method="wilcox",
                        method="auto"
                    ).pvalue
                )

        except Exception:
            pval = 1.0

        # Specificity against ALL annotated cell types.
        if (
            tf in all_celltype_mean_df.index and
            target_ct in all_celltype_mean_df.columns
        ):
            all_ct_target_mean = float(
                all_celltype_mean_df.at[tf, target_ct]
            )

            all_ct_total = float(all_celltype_total.at[tf])

            all_celltype_specificity = (
                all_ct_target_mean / all_ct_total
                if np.isfinite(all_ct_total) and all_ct_total > 0
                else np.nan
            )
        else:
            all_celltype_specificity = np.nan

        results.append({
            "gene": tf,
            "cell_type": target_ct,
            "n_paired_samples": len(target_values),
            "mean_target": mean_target,
            "mean_other": mean_other,
            "log2FC": log2fc,
            "AUC": auc,
            "pval": pval,
            "all_celltype_specificity": all_celltype_specificity
        })


# 3. Assemble results and calculate adjusted p-values
df_res = pd.DataFrame(results)

if df_res.empty:
    raise ValueError("No TF specificity results were produced.")

df_res["pval_adj"] = np.nan

# BH correction separately within each target cell type.
for ct in TARGET_CELL_TYPES:

    mask = df_res["cell_type"] == ct

    if mask.any():
        df_res.loc[mask, "pval_adj"] = multipletests(
            df_res.loc[mask, "pval"].astype(float),
            method="fdr_bh"
        )[1]


# 4. Calculate the three specificity/ranking approaches

# A. Original specificity:
# Relative expression between Stromal and Epithelial populations.

target_mean_sum = (
    df_res.groupby("gene")["mean_target"].transform("sum")
)

df_res["original_specificity"] = (
    df_res["mean_target"] /
    target_mean_sum.replace(0, np.nan)
)


# B. All-cell-type specificity:
# Already calculated against all annotated cell types above.


# C. Base score without a specificity multiplier.

df_res["base_score"] = (
    (df_res["AUC"] - 0.5) * 2
    * np.maximum(0, df_res["log2FC"])
)


# 5. Calculate all three importance scores
df_res["importance_score_original"] = (
    df_res["base_score"]
    * (df_res["original_specificity"] * 4)
)

# Revised formula using all-cell-type specificity.
df_res["importance_score_all_celltypes"] = (
    df_res["base_score"]
    * (df_res["all_celltype_specificity"] * 4)
)

# No-specificity version.
df_res["importance_score_no_specificity"] = (
    df_res["base_score"]
)

# Retain the original column name for compatibility
# with downstream code that uses importance_score.
df_res["importance_score"] = (
    df_res["importance_score_original"]
)


# 6. Compare rankings within each target cell type
score_columns = {
    "original": "importance_score_original",
    "all_celltypes": "importance_score_all_celltypes",
    "no_specificity": "importance_score_no_specificity"
}

for rank_name, score_col in score_columns.items():

    df_res[f"rank_{rank_name}"] = (
        df_res.groupby("cell_type")[score_col]
        .rank(method="min", ascending=False)
    )

ranking_columns = [
    "gene",
    "cell_type",
    "n_paired_samples",
    "mean_target",
    "mean_other",
    "log2FC",
    "AUC",
    "pval",
    "pval_adj",
    "original_specificity",
    "all_celltype_specificity",
    "importance_score_original",
    "importance_score_all_celltypes",
    "importance_score_no_specificity",
    "rank_original",
    "rank_all_celltypes",
    "rank_no_specificity"
]

ranking_comparison = (
    df_res[ranking_columns]
    .sort_values(["cell_type", "rank_original", "gene"])
    .reset_index(drop=True)
)

# 7. Export CSV and Excel
TF_RESULTS_PATH = (
    PSEUDOBULK_DIR / "TF_specificity_results_comparison.csv"
)

EXCEL_RESULTS_PATH = (
    PSEUDOBULK_DIR / "TF_importance_score_comparison.xlsx"
)

df_res.to_csv(TF_RESULTS_PATH, index=False)

method_notes = pd.DataFrame({
    "Metric": [
        "Original specificity",
        "All-cell-type specificity",
        "No-specificity score",
        "AUC",
        "log2FC",
        "Adjusted p-value"
    ],
    "Definition": [
        "Mean target expression divided by the sum of mean expression across Stromal and Epithelial target populations.",
        "Mean target expression divided by the sum of mean expression across all annotated cell types.",
        "2 × (AUC - 0.5) × max(0, log2FC).",
        "Descriptive discrimination between target and reference expression profiles.",
        "Log2 ratio of mean target expression to mean reference expression, with a 0.001 pseudocount.",
        "Benjamini-Hochberg correction applied separately within each target cell type."
    ],
    "Interpretation": [
        "Relative preference for one endometrial target population over the other.",
        "Relative expression allocation to the target population across all included cell types.",
        "Expression-based ranking without a specificity multiplier.",
        "Not a paired inferential test or a measure of TF regulatory activity.",
        "Positive values indicate higher mean expression in the target population.",
        "Controls the false discovery rate within each target-cell-type analysis."
    ]
})

with pd.ExcelWriter(
    EXCEL_RESULTS_PATH,
    engine="openpyxl"
) as writer:

    df_res.to_excel(
        writer,
        sheet_name="All_results",
        index=False
    )

    ranking_comparison.to_excel(
        writer,
        sheet_name="Ranking_comparison",
        index=False
    )

    method_notes.to_excel(
        writer,
        sheet_name="Method_notes",
        index=False
    )

    for ct in TARGET_CELL_TYPES:

        ct_results = (
            df_res[df_res["cell_type"] == ct]
            .sort_values(
                "importance_score_original",
                ascending=False
            )
        )

        if not ct_results.empty:
            ct_results.to_excel(
                writer,
                sheet_name=ct,
                index=False
            )


# 8. Display top 20 TFs under each scoring approach
for ct in TARGET_CELL_TYPES:

    print(f"\n{'=' * 65}")
    print(f"{ct.upper()} TF RANKINGS")
    print(f"{'=' * 65}")

    ct_df = df_res[df_res["cell_type"] == ct]

    for rank_name, score_col in score_columns.items():

        print(f"\nTop 20 — {rank_name.replace('_', ' ').title()}")

        display(
            ct_df.sort_values(
                score_col,
                ascending=False
            )[
                [
                    "gene",
                    "mean_target",
                    "log2FC",
                    "AUC",
                    "pval_adj",
                    "original_specificity",
                    "all_celltype_specificity",
                    score_col
                ]
            ].head(20)
        )


print("\nAnalysis completed.")
print(f"CSV saved: {TF_RESULTS_PATH}")
print(f"Excel comparison saved: {EXCEL_RESULTS_PATH}")


In [ ]:
# 5C. Generate volcano plots
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# --- FIX 1: Define the output directory safely ---
# If FIG_DIR is defined in your environment, it uses it. Otherwise, it creates a local folder.
VOLCANO_DIR = Path("Volcano_Plots")
VOLCANO_DIR.mkdir(parents=True, exist_ok=True)

def plot_tf_volcano(results, cell_type, color):
    subset = results[results["cell_type"] == cell_type].copy()
    subset = subset[
        np.isfinite(subset["log2FC"])
        & np.isfinite(subset["pval_adj"])
    ].copy()

    if subset.empty:
        print(f"No valid TF results for {cell_type}; volcano plot skipped.")
        return

    subset["neg_log10_padj"] = -np.log10(
        subset["pval_adj"].clip(lower=1e-300)
    )
    
    # --- INDUSTRY STANDARD THRESHOLDS ---
    subset["significant"] = (
        (subset["pval_adj"] < 0.05)
        & (subset["log2FC"].abs() > 1.0)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Non-significant points (light grey)
    ax.scatter(
        subset.loc[~subset["significant"], "log2FC"],
        subset.loc[~subset["significant"], "neg_log10_padj"],
        color="#e0e0e0", alpha=0.5, s=12, edgecolors="none"
    )
    
    # Significant points
    sig = subset[subset["significant"]]
    ax.scatter(
        sig["log2FC"], sig["neg_log10_padj"],
        color=color, alpha=0.7, s=18, edgecolors="none",
        label=f"sig+strong ({len(sig)})"
    )

    # Threshold lines (Standard: p=0.05, |log2FC|=1)
    ax.axhline(-np.log10(0.05), color="gray", linestyle="--", linewidth=0.8)
    ax.axvline(1.0, color="gray", linestyle="--", linewidth=0.8)
    ax.axvline(-1.0, color="gray", linestyle="--", linewidth=0.8)

    # Label the top 10 most significant TFs
    top10 = sig.nsmallest(min(10, len(sig)), "pval_adj")
    top10 = top10.iloc[top10["log2FC"].abs().argsort()[::-1]]

    # --- SMART MANUAL LABEL PLACEMENT ---
    placed_positions = list(zip(subset["log2FC"], subset["neg_log10_padj"]))
    max_label_y = subset["neg_log10_padj"].max()

    for _, row in top10.iterrows():
        x = row["log2FC"]
        y = row["neg_log10_padj"]

        sign = 1 if x > 0 else -1
        tx = x + (0.2 * sign)
        ty = y + 0.1 

        # Collision loop: Push text up and outwards until it's clear
        while True:
            overlap = False
            for px, py in placed_positions:
                if abs(tx - px) < 0.8 and abs(ty - py) < 0.25:
                    ty += 0.15  
                    tx += (0.05 * sign)  
                    overlap = True
                    break
            if not overlap:
                break

        placed_positions.append((tx, ty))
        max_label_y = max(max_label_y, ty)

        # Draw annotation (Removed fontweight="medium" to fix the error)
        ax.annotate(
            str(row["gene"]),
            xy=(x, y),
            xytext=(tx, ty),
            fontsize=7.5,
            color="#333333",
            arrowprops=dict(
                arrowstyle="-", 
                color="#666666", 
                lw=0.6, 
                shrinkA=0, 
                shrinkB=2
            ),
            clip_on=True
        )

    # Set limits dynamically to ensure labels are inside the border
    y_max = max_label_y + 0.5
    x_min, x_max = subset["log2FC"].min(), subset["log2FC"].max()
    
    ax.set_ylim(-0.5, y_max)
    ax.set_xlim(x_min - 1, x_max + 1)

    # Professional Aesthetics
    ax.set_title(f"{cell_type}-specific TFs", fontweight="bold", fontsize=12)
    ax.set_xlabel("log2 Fold Change", fontsize=10)
    ax.set_ylabel("-log10(adj. p-value)", fontsize=10)
    ax.grid(True, linestyle='-', alpha=0.15, zorder=0)
    ax.legend(frameon=False, loc="upper left")
    
    fig.tight_layout()

    # Save outputs
    safe_name = cell_type.lower()
    for ext in ["png", "pdf"]:
        out = VOLCANO_DIR / f"Volcano_{safe_name}_TFs.{ext}"
        fig.savefig(out, dpi=300, bbox_inches="tight")
    
    print(f"Successfully saved {cell_type} plot to {out.parent}")
    plt.show()
    plt.close(fig)

# --- RUN THE PLOTS ---
# Make sure df_res exists before running this line!
plot_tf_volcano(df_res, "Epithelial", "#619CFF")
plot_tf_volcano(df_res, "Stromal", "#F8766D")

In [ ]:
# 5D. Heatmap of candidate TF expression across cell types
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Select significant candidates first. If none pass the thresholds, show the
# highest-ranked TFs so the heatmap remains informative without calling them significant.
significant = df_res[
    (df_res["pval_adj"] < 0.05)
    & (df_res["log2FC"] > 1.0)
].copy()

if not significant.empty:
    candidate_genes = (
        significant.sort_values("importance_score", ascending=False)
        .groupby("cell_type", observed=True)
        .head(20)["gene"].drop_duplicates().tolist()
    )
    heatmap_title = "Significant candidate TFs"
else:
    candidate_genes = (
        df_res.sort_values("importance_score", ascending=False)
        .groupby("cell_type", observed=True)
        .head(15)["gene"].drop_duplicates().tolist()
    )
    heatmap_title = "Top-ranked candidate TFs (no TFs passed significance thresholds)"

# Mean normalized expression across sample-level pseudobulk profiles for each cell type.
available_types = [
    ct for ct in ["Stromal", "Epithelial", "Immune", "Endothelial"]
    if ct in set(pb_metadata["cell_type"])
]
mean_by_celltype = {}
for ct in available_types:
    cols = pb_metadata.loc[
        pb_metadata["cell_type"] == ct, "pseudobulk_sample"
    ].tolist()
    if cols:
        mean_by_celltype[ct] = pb_norm[cols].mean(axis=1)

heat = pd.DataFrame(mean_by_celltype).reindex(index=candidate_genes)
heat = heat.dropna(how="all").fillna(0)

if heat.empty:
    print("No candidate TFs available for heatmap.")
else:
    # Row-wise z-scores highlight relative cell-type enrichment per TF.
    row_mean = heat.mean(axis=1)
    row_sd = heat.std(axis=1).replace(0, np.nan)
    heat_z = heat.sub(row_mean, axis=0).div(row_sd, axis=0).fillna(0)

    fig_height = max(5, 0.28 * len(heat_z))
    fig, ax = plt.subplots(figsize=(8, fig_height))
    sns.heatmap(
        heat_z,
        cmap="vlag",
        center=0,
        linewidths=0.25,
        linecolor="white",
        cbar_kws={"label": "Row z-score of mean normalized expression"},
        ax=ax
    )
    ax.set_title(heatmap_title, fontweight="bold")
    ax.set_xlabel("Cell type")
    ax.set_ylabel("Transcription factor")
    fig.tight_layout()

    for ext in ["png", "pdf"]:
        fig.savefig(FIG_DIR / f"TF_specificity_heatmap.{ext}",
                    dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    heat.to_csv(PSEUDOBULK_DIR / "TF_heatmap_mean_expression.csv")
    print("Saved TF specificity heatmap and its underlying expression matrix.")


In [ ]:
# 5E. Export significant TF tables to Excel
import numpy as np
import pandas as pd

TF_EXPORT_PATH = PSEUDOBULK_DIR / "TF_Filtered_Significant.xlsx"

df_filtered = df_res[
    (df_res["pval_adj"] < 0.05)
    & (df_res["log2FC"] > 1.0)
].copy()

df_filtered["neg_log10_pval_adj"] = -np.log10(
    df_filtered["pval_adj"].clip(lower=1e-300)
)

epi_sig = (
    df_filtered[df_filtered["cell_type"] == "Epithelial"]
    .sort_values("importance_score", ascending=False)
)
stromal_sig = (
    df_filtered[df_filtered["cell_type"] == "Stromal"]
    .sort_values("importance_score", ascending=False)
)

with pd.ExcelWriter(TF_EXPORT_PATH, engine="openpyxl") as writer:
    epi_sig.to_excel(writer, sheet_name="Epithelial_Sig", index=False)
    stromal_sig.to_excel(writer, sheet_name="Stromal_Sig", index=False)
    df_res.to_excel(writer, sheet_name="All_TF_Results", index=False)

print(f"Excel export: {TF_EXPORT_PATH}")
print(f"Epithelial significant TFs: {len(epi_sig)}")
print(f"Stromal significant TFs: {len(stromal_sig)}")


## 6. QC Plots

In [ ]:
# 6A. Setup
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

FIG_DIR = Path("Figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load the already QC-filtered dataset
adata = sc.read_h5ad("endometrium_final.h5ad")

# Check sample information
sample_order = sorted(adata.obs["sample_id"].unique())

print(f"Total cells: {adata.n_obs:,}")
print(f"Number of samples: {len(sample_order)}")

In [ ]:
# 6B. Post-QC: Number of genes per cell
metric = "n_genes_by_counts"

fig, ax = plt.subplots(figsize=(20, 6))

sns.violinplot(
    data=adata.obs,
    x="sample_id",
    y=metric,
    order=sample_order,
    palette=sns.color_palette("husl", len(sample_order)),
    inner=None,
    cut=0,
    linewidth=0.8,
    ax=ax
)

ax.set_title("Post-QC: n_genes per cell", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("n_genes")
ax.tick_params(axis="x", rotation=90)

fig.tight_layout()
fig.savefig(FIG_DIR / "QC_post_n_genes.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "QC_post_n_genes.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 6C. Post-QC: Number of UMIs per cell
metric = "total_counts"

fig, ax = plt.subplots(figsize=(20, 6))

sns.violinplot(
    data=adata.obs,
    x="sample_id",
    y=metric,
    order=sample_order,
    palette=sns.color_palette("husl", len(sample_order)),
    inner=None,
    cut=0,
    linewidth=0.8,
    ax=ax
)

ax.set_title("Post-QC: n_UMIs per cell", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("n_UMIs")
ax.tick_params(axis="x", rotation=90)

fig.tight_layout()
fig.savefig(FIG_DIR / "QC_post_n_UMIs.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "QC_post_n_UMIs.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# 6D. Post-QC: Percent mitochondrial counts per cell
metric = "pct_counts_mt"

fig, ax = plt.subplots(figsize=(20, 6))

sns.violinplot(
    data=adata.obs,
    x="sample_id",
    y=metric,
    order=sample_order,
    palette=sns.color_palette("husl", len(sample_order)),
    inner=None,
    cut=0,
    linewidth=0.8,
    ax=ax
)

ax.set_title("Post-QC: % mitochondrial UMIs", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("pct_counts_mt")
ax.tick_params(axis="x", rotation=90)

fig.tight_layout()
fig.savefig(FIG_DIR / "QC_post_pct_MT.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "QC_post_pct_MT.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)